# KLA Inventory Optimization — PuLP + cuOpt

**Two-stage optimization for NLO Crystal Assembly (994-023) across 4 global warehouses serving 5 active escalation cases.**

- **Stage 1 — PuLP (CPU):** Multi-echelon MIP — decides warehouse-to-case assignments and inter-warehouse transfers
- **Stage 2 — cuOpt (GPU):** VRP with Time Windows — optimizes delivery sequence respecting SLA deadlines

Use the parameter controls below to model different scenarios and see impact analysis.

In [ ]:
import os
import json
import math
import uuid
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import pulp
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

warnings.filterwarnings("ignore")

try:
    import snowflake.connector
    SF_AVAILABLE = True
except ImportError:
    SF_AVAILABLE = False
    print("snowflake-connector-python not installed — using local demo data")

print(f"PuLP {pulp.__version__} | CBC solver ready")

## Cell 2: Load Data from Snowflake (or local fallback)

In [ ]:
UNIT_COST = 45000.0
VAT_RATE = 0.10

def load_from_snowflake():
    conn = snowflake.connector.connect(
        connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "mydemo"
    )
    cur = conn.cursor()
    cur.execute("SELECT * FROM KLA_SUPPLY_CHAIN.APP.WAREHOUSE_INVENTORY")
    inv_df = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    cur.execute("SELECT * FROM KLA_SUPPLY_CHAIN.APP.ESCALATION_DEMANDS")
    dem_df = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    cur.execute("SELECT * FROM KLA_SUPPLY_CHAIN.APP.TARIFF_RATES")
    tar_df = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    cur.close()
    conn.close()
    return inv_df, dem_df, tar_df

def load_local_fallback():
    inv_df = pd.DataFrame([
        {"WAREHOUSE_ID": "WH-TUCSON",    "WAREHOUSE_CITY": "Tucson",    "COUNTRY_CODE": "USA", "PART_ID": "994-023", "QTY_ON_HAND": 3, "QTY_IN_TRANSIT": 1, "QTY_BACKORDERED": 0, "REORDER_POINT": 2, "BATCH_LOT": "B-2024-A", "HOLDING_COST_DAILY": 12.50},
        {"WAREHOUSE_ID": "WH-SANJOSE",   "WAREHOUSE_CITY": "San Jose",  "COUNTRY_CODE": "USA", "PART_ID": "994-023", "QTY_ON_HAND": 0, "QTY_IN_TRANSIT": 2, "QTY_BACKORDERED": 4, "REORDER_POINT": 3, "BATCH_LOT": "B-2024-A", "HOLDING_COST_DAILY": 8.75},
        {"WAREHOUSE_ID": "WH-SINGAPORE", "WAREHOUSE_CITY": "Singapore", "COUNTRY_CODE": "SGP", "PART_ID": "994-023", "QTY_ON_HAND": 2, "QTY_IN_TRANSIT": 0, "QTY_BACKORDERED": 0, "REORDER_POINT": 2, "BATCH_LOT": "B-2024-B", "HOLDING_COST_DAILY": 15.00},
        {"WAREHOUSE_ID": "WH-DRESDEN",   "WAREHOUSE_CITY": "Dresden",   "COUNTRY_CODE": "DEU", "PART_ID": "994-023", "QTY_ON_HAND": 1, "QTY_IN_TRANSIT": 1, "QTY_BACKORDERED": 0, "REORDER_POINT": 2, "BATCH_LOT": "B-2024-C", "HOLDING_COST_DAILY": 11.25},
    ])
    dem_df = pd.DataFrame([
        {"CASE_ID": "ESC-2026-4281", "CUSTOMER": "Samsung",  "DESTINATION_CITY": "Pyeongtaek",  "DESTINATION_LAT": 36.99,  "DESTINATION_LON": 127.11,  "PART_ID": "994-023", "SEVERITY": "SEV1", "SLA_REMAINING_HRS": -72,  "REVENUE_AT_RISK": 4200000, "SLA_PENALTY_RATE": 50000},
        {"CASE_ID": "ESC-2026-4305", "CUSTOMER": "Renesas",  "DESTINATION_CITY": "Hitachinaka",  "DESTINATION_LAT": 36.40,  "DESTINATION_LON": 140.53,  "PART_ID": "994-023", "SEVERITY": "SEV1", "SLA_REMAINING_HRS": -24,  "REVENUE_AT_RISK": 1800000, "SLA_PENALTY_RATE": 50000},
        {"CASE_ID": "ESC-2026-4198", "CUSTOMER": "TSMC",     "DESTINATION_CITY": "Taichung",     "DESTINATION_LAT": 24.15,  "DESTINATION_LON": 120.67,  "PART_ID": "994-023", "SEVERITY": "SEV2", "SLA_REMAINING_HRS": -120, "REVENUE_AT_RISK": 2400000, "SLA_PENALTY_RATE": 25000},
        {"CASE_ID": "ESC-2026-4312", "CUSTOMER": "SK Hynix", "DESTINATION_CITY": "Icheon",       "DESTINATION_LAT": 37.28,  "DESTINATION_LON": 127.44,  "PART_ID": "994-023", "SEVERITY": "SEV2", "SLA_REMAINING_HRS": 0,    "REVENUE_AT_RISK": 1200000, "SLA_PENALTY_RATE": 25000},
        {"CASE_ID": "ESC-2026-4287", "CUSTOMER": "Intel",    "DESTINATION_CITY": "Chandler",     "DESTINATION_LAT": 33.308, "DESTINATION_LON": -111.843,"PART_ID": "994-023", "SEVERITY": "SEV3", "SLA_REMAINING_HRS": 24,   "REVENUE_AT_RISK": 600000,  "SLA_PENALTY_RATE": 10000},
    ])
    tar_df = pd.DataFrame([
        {"ORIGIN_COUNTRY": "USA", "DEST_COUNTRY": "KOR", "TARIFF_RATE_PCT": 0.08,  "FTA_AGREEMENT": None,              "SHIPPING_DAYS": 2, "BASE_FREIGHT": 4500},
        {"ORIGIN_COUNTRY": "SGP", "DEST_COUNTRY": "KOR", "TARIFF_RATE_PCT": 0.00,  "FTA_AGREEMENT": "ASEAN-Korea FTA", "SHIPPING_DAYS": 1, "BASE_FREIGHT": 1800},
        {"ORIGIN_COUNTRY": "DEU", "DEST_COUNTRY": "KOR", "TARIFF_RATE_PCT": 0.032, "FTA_AGREEMENT": "EU-Korea FTA",    "SHIPPING_DAYS": 2, "BASE_FREIGHT": 3200},
        {"ORIGIN_COUNTRY": "USA", "DEST_COUNTRY": "JPN", "TARIFF_RATE_PCT": 0.05,  "FTA_AGREEMENT": None,              "SHIPPING_DAYS": 2, "BASE_FREIGHT": 4200},
        {"ORIGIN_COUNTRY": "SGP", "DEST_COUNTRY": "JPN", "TARIFF_RATE_PCT": 0.00,  "FTA_AGREEMENT": "CPTPP",           "SHIPPING_DAYS": 1, "BASE_FREIGHT": 2100},
        {"ORIGIN_COUNTRY": "DEU", "DEST_COUNTRY": "JPN", "TARIFF_RATE_PCT": 0.038, "FTA_AGREEMENT": "EU-Japan EPA",    "SHIPPING_DAYS": 2, "BASE_FREIGHT": 3400},
        {"ORIGIN_COUNTRY": "USA", "DEST_COUNTRY": "TWN", "TARIFF_RATE_PCT": 0.06,  "FTA_AGREEMENT": None,              "SHIPPING_DAYS": 2, "BASE_FREIGHT": 4800},
        {"ORIGIN_COUNTRY": "SGP", "DEST_COUNTRY": "TWN", "TARIFF_RATE_PCT": 0.00,  "FTA_AGREEMENT": "ASEAN-Taiwan",    "SHIPPING_DAYS": 1, "BASE_FREIGHT": 1900},
        {"ORIGIN_COUNTRY": "DEU", "DEST_COUNTRY": "TWN", "TARIFF_RATE_PCT": 0.035, "FTA_AGREEMENT": None,              "SHIPPING_DAYS": 2, "BASE_FREIGHT": 3500},
        {"ORIGIN_COUNTRY": "USA", "DEST_COUNTRY": "USA", "TARIFF_RATE_PCT": 0.00,  "FTA_AGREEMENT": "Domestic",        "SHIPPING_DAYS": 1, "BASE_FREIGHT": 400},
        {"ORIGIN_COUNTRY": "SGP", "DEST_COUNTRY": "SGP", "TARIFF_RATE_PCT": 0.00,  "FTA_AGREEMENT": "Domestic",        "SHIPPING_DAYS": 0, "BASE_FREIGHT": 200},
        {"ORIGIN_COUNTRY": "DEU", "DEST_COUNTRY": "DEU", "TARIFF_RATE_PCT": 0.00,  "FTA_AGREEMENT": "Domestic",        "SHIPPING_DAYS": 0, "BASE_FREIGHT": 250},
        {"ORIGIN_COUNTRY": "USA", "DEST_COUNTRY": "BEL", "TARIFF_RATE_PCT": 0.04,  "FTA_AGREEMENT": None,              "SHIPPING_DAYS": 2, "BASE_FREIGHT": 3800},
        {"ORIGIN_COUNTRY": "SGP", "DEST_COUNTRY": "BEL", "TARIFF_RATE_PCT": 0.025, "FTA_AGREEMENT": None,              "SHIPPING_DAYS": 2, "BASE_FREIGHT": 4100},
        {"ORIGIN_COUNTRY": "DEU", "DEST_COUNTRY": "BEL", "TARIFF_RATE_PCT": 0.00,  "FTA_AGREEMENT": "EU Single Market","SHIPPING_DAYS": 1, "BASE_FREIGHT": 350},
    ])
    return inv_df, dem_df, tar_df

CITY_TO_COUNTRY = {
    "Pyeongtaek": "KOR", "Hitachinaka": "JPN", "Taichung": "TWN",
    "Icheon": "KOR", "Chandler": "USA", "Leuven": "BEL",
    "Tucson": "USA", "San Jose": "USA", "Singapore": "SGP", "Dresden": "DEU",
}

if SF_AVAILABLE:
    try:
        inv_df, dem_df, tar_df = load_from_snowflake()
        print("Loaded data from Snowflake")
    except Exception as e:
        print(f"Snowflake load failed ({e}), using local fallback")
        inv_df, dem_df, tar_df = load_local_fallback()
else:
    inv_df, dem_df, tar_df = load_local_fallback()
    print("Using local demo data")

warehouses = inv_df["WAREHOUSE_ID"].tolist()
cases = dem_df["CASE_ID"].tolist()

print(f"\nWarehouses: {len(warehouses)} | Cases: {len(cases)} | Tariff corridors: {len(tar_df)}")
display(inv_df[["WAREHOUSE_ID","WAREHOUSE_CITY","QTY_ON_HAND","QTY_IN_TRANSIT","QTY_BACKORDERED","BATCH_LOT"]])
display(dem_df[["CASE_ID","CUSTOMER","SEVERITY","SLA_REMAINING_HRS","REVENUE_AT_RISK"]])

## Cell 3: Scenario Parameters (ipywidgets)

In [ ]:
style = {"description_width": "180px"}
layout = widgets.Layout(width="500px")

sla_sev1_penalty = widgets.FloatSlider(
    value=50000, min=10000, max=500000, step=5000,
    description="SEV1 Penalty ($/hr):", style=style, layout=layout
)
sla_sev2_penalty = widgets.FloatSlider(
    value=25000, min=5000, max=250000, step=2500,
    description="SEV2 Penalty ($/hr):", style=style, layout=layout
)
sla_sev3_penalty = widgets.FloatSlider(
    value=10000, min=1000, max=100000, step=1000,
    description="SEV3 Penalty ($/hr):", style=style, layout=layout
)
shipping_multiplier = widgets.FloatSlider(
    value=1.0, min=0.5, max=3.0, step=0.1,
    description="Shipping Cost Mult:", style=style, layout=layout
)
transfer_cost_multiplier = widgets.FloatSlider(
    value=1.0, min=0.5, max=3.0, step=0.1,
    description="Transfer Cost Mult:", style=style, layout=layout
)
customer_priority = widgets.Dropdown(
    options=["All Equal"] + sorted(dem_df["CUSTOMER"].unique().tolist()),
    value="All Equal",
    description="Prioritize Customer:", style=style, layout=layout
)
safety_stock = widgets.IntSlider(
    value=0, min=0, max=2, step=1,
    description="Min Safety Stock:", style=style, layout=layout
)
batch_exclusion = widgets.Checkbox(
    value=True, description="Exclude Batch B-2024-X",
    style=style, layout=layout
)

output_area = widgets.Output()
run_button = widgets.Button(
    description="  Run Optimization",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="220px", height="40px"),
)

param_box = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 8px 0'>SLA Penalty Rates</h3>"),
    sla_sev1_penalty, sla_sev2_penalty, sla_sev3_penalty,
    widgets.HTML("<h3 style='margin:12px 0 8px 0'>Cost Multipliers</h3>"),
    shipping_multiplier, transfer_cost_multiplier,
    widgets.HTML("<h3 style='margin:12px 0 8px 0'>Constraints</h3>"),
    customer_priority, safety_stock, batch_exclusion,
    widgets.HTML("<br>"),
    run_button,
])

display(param_box)

## Cell 4: PuLP Inventory Allocation MIP Solver

In [ ]:
INTER_WH_FREIGHT = {
    ("WH-TUCSON", "WH-SANJOSE"):   400,
    ("WH-SANJOSE", "WH-TUCSON"):   400,
    ("WH-TUCSON", "WH-SINGAPORE"): 5200,
    ("WH-SINGAPORE", "WH-TUCSON"): 5200,
    ("WH-TUCSON", "WH-DRESDEN"):   4800,
    ("WH-DRESDEN", "WH-TUCSON"):   4800,
    ("WH-SANJOSE", "WH-SINGAPORE"):5400,
    ("WH-SINGAPORE", "WH-SANJOSE"):5400,
    ("WH-SANJOSE", "WH-DRESDEN"):  5000,
    ("WH-DRESDEN", "WH-SANJOSE"):  5000,
    ("WH-SINGAPORE", "WH-DRESDEN"):4200,
    ("WH-DRESDEN", "WH-SINGAPORE"):4200,
}

def get_landed_cost(wh_country, dest_city, ship_mult=1.0):
    dest_country = CITY_TO_COUNTRY.get(dest_city, "USA")
    row = tar_df[(tar_df["ORIGIN_COUNTRY"] == wh_country) & (tar_df["DEST_COUNTRY"] == dest_country)]
    if row.empty:
        return UNIT_COST + 5000
    r = row.iloc[0]
    freight = float(r["BASE_FREIGHT"]) * ship_mult
    duty = UNIT_COST * float(r["TARIFF_RATE_PCT"])
    vat = (UNIT_COST + duty) * VAT_RATE
    return UNIT_COST + freight + duty + vat

def get_shipping_days(wh_country, dest_city):
    dest_country = CITY_TO_COUNTRY.get(dest_city, "USA")
    row = tar_df[(tar_df["ORIGIN_COUNTRY"] == wh_country) & (tar_df["DEST_COUNTRY"] == dest_country)]
    if row.empty:
        return 3
    return int(row.iloc[0]["SHIPPING_DAYS"])

def solve_pulp(params):
    sev_penalties = {
        "SEV1": params["sla_sev1_penalty"],
        "SEV2": params["sla_sev2_penalty"],
        "SEV3": params["sla_sev3_penalty"],
    }
    ship_mult = params["shipping_multiplier"]
    xfer_mult = params["transfer_cost_multiplier"]
    priority_customer = params["customer_priority"]
    min_safety = params["safety_stock"]
    exclude_batch = params["batch_exclusion"]

    prob = pulp.LpProblem("KLA_Inventory_Optimization", pulp.LpMinimize)

    assign = {}
    for w in warehouses:
        for k in cases:
            assign[(w, k)] = pulp.LpVariable(f"assign_{w}_{k}", cat="Binary")

    transfer = {}
    for wi in warehouses:
        for wj in warehouses:
            if wi != wj:
                transfer[(wi, wj)] = pulp.LpVariable(
                    f"transfer_{wi}_{wj}", lowBound=0, cat="Integer"
                )

    unfulfilled = {}
    for k in cases:
        unfulfilled[k] = pulp.LpVariable(f"unfulfilled_{k}", cat="Binary")

    landed_costs = {}
    for w in warehouses:
        wh_row = inv_df[inv_df["WAREHOUSE_ID"] == w].iloc[0]
        wh_country = wh_row["COUNTRY_CODE"]
        for k in cases:
            case_row = dem_df[dem_df["CASE_ID"] == k].iloc[0]
            landed_costs[(w, k)] = get_landed_cost(wh_country, case_row["DESTINATION_CITY"], ship_mult)

    obj = pulp.lpSum([
        landed_costs[(w, k)] * assign[(w, k)]
        for w in warehouses for k in cases
    ])

    for wi in warehouses:
        for wj in warehouses:
            if wi != wj:
                base = INTER_WH_FREIGHT.get((wi, wj), 5000)
                obj += base * xfer_mult * transfer[(wi, wj)]

    for k in cases:
        case_row = dem_df[dem_df["CASE_ID"] == k].iloc[0]
        sev = case_row["SEVERITY"]
        base_penalty = sev_penalties.get(sev, 10000)
        breach_hrs = max(0, -int(case_row["SLA_REMAINING_HRS"]))
        penalty = base_penalty * max(1, breach_hrs)
        if priority_customer != "All Equal" and case_row["CUSTOMER"] == priority_customer:
            penalty *= 2.0
        obj += penalty * unfulfilled[k]

    for w in warehouses:
        wh_row = inv_df[inv_df["WAREHOUSE_ID"] == w].iloc[0]
        obj += float(wh_row["HOLDING_COST_DAILY"]) * 30 * pulp.lpSum([
            pulp.LpVariable(f"end_stock_{w}", lowBound=0)
        ])

    prob += obj

    for k in cases:
        prob += (
            pulp.lpSum([assign[(w, k)] for w in warehouses]) + unfulfilled[k] == 1,
            f"fulfill_or_miss_{k}"
        )

    for w in warehouses:
        wh_row = inv_df[inv_df["WAREHOUSE_ID"] == w].iloc[0]
        stock = int(wh_row["QTY_ON_HAND"])

        outbound_transfers = pulp.lpSum([
            transfer[(w, wj)] for wj in warehouses if wj != w
        ])
        inbound_transfers = pulp.lpSum([
            transfer[(wi, w)] for wi in warehouses if wi != w
        ])
        assignments_out = pulp.lpSum([assign[(w, k)] for k in cases])

        ending = stock + inbound_transfers - outbound_transfers - assignments_out
        prob += (ending >= min_safety, f"safety_stock_{w}")

    if exclude_batch:
        for w in warehouses:
            wh_row = inv_df[inv_df["WAREHOUSE_ID"] == w].iloc[0]
            if wh_row["BATCH_LOT"] == "B-2024-X":
                for k in cases:
                    prob += (assign[(w, k)] == 0, f"batch_exclude_{w}_{k}")
                for wj in warehouses:
                    if wj != w and (w, wj) in transfer:
                        prob += (transfer[(w, wj)] == 0, f"batch_xfer_exclude_{w}_{wj}")

    for wi in warehouses:
        for wj in warehouses:
            if wi != wj and (wi, wj) in transfer:
                wh_row = inv_df[inv_df["WAREHOUSE_ID"] == wi].iloc[0]
                prob += (
                    transfer[(wi, wj)] <= int(wh_row["QTY_ON_HAND"]),
                    f"max_transfer_{wi}_{wj}"
                )

    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    results = {
        "status": pulp.LpStatus[prob.status],
        "total_cost": pulp.value(prob.objective),
        "assignments": [],
        "transfers": [],
        "unfulfilled": [],
    }

    for w in warehouses:
        for k in cases:
            if pulp.value(assign[(w, k)]) > 0.5:
                case_row = dem_df[dem_df["CASE_ID"] == k].iloc[0]
                wh_row = inv_df[inv_df["WAREHOUSE_ID"] == w].iloc[0]
                results["assignments"].append({
                    "warehouse": w,
                    "warehouse_city": wh_row["WAREHOUSE_CITY"],
                    "case_id": k,
                    "customer": case_row["CUSTOMER"],
                    "destination": case_row["DESTINATION_CITY"],
                    "severity": case_row["SEVERITY"],
                    "landed_cost": round(landed_costs[(w, k)], 2),
                    "shipping_days": get_shipping_days(wh_row["COUNTRY_CODE"], case_row["DESTINATION_CITY"]),
                })

    for wi in warehouses:
        for wj in warehouses:
            if wi != wj and (wi, wj) in transfer:
                val = pulp.value(transfer[(wi, wj)])
                if val and val > 0.5:
                    results["transfers"].append({
                        "from": wi,
                        "to": wj,
                        "quantity": int(val),
                        "cost": round(INTER_WH_FREIGHT.get((wi, wj), 5000) * xfer_mult * val, 2),
                    })

    for k in cases:
        if pulp.value(unfulfilled[k]) > 0.5:
            case_row = dem_df[dem_df["CASE_ID"] == k].iloc[0]
            results["unfulfilled"].append({
                "case_id": k,
                "customer": case_row["CUSTOMER"],
                "severity": case_row["SEVERITY"],
            })

    fulfilled = len(results["assignments"])
    total_cases = len(cases)
    results["sla_coverage"] = round(fulfilled / total_cases * 100, 1)

    cost_breakdown = {"shipping": 0, "transfer": 0, "sla_penalty": 0}
    for a in results["assignments"]:
        cost_breakdown["shipping"] += a["landed_cost"]
    for t in results["transfers"]:
        cost_breakdown["transfer"] += t["cost"]
    for u in results["unfulfilled"]:
        sev = u["severity"]
        case_row = dem_df[dem_df["CASE_ID"] == u["case_id"]].iloc[0]
        breach = max(0, -int(case_row["SLA_REMAINING_HRS"]))
        cost_breakdown["sla_penalty"] += sev_penalties.get(sev, 10000) * max(1, breach)
    results["cost_breakdown"] = cost_breakdown

    return results

print("PuLP solver ready")

## Cell 5: cuOpt Shipment Routing (VRP with Time Windows)

In [ ]:
WH_COORDS = {
    "WH-TUCSON":    (32.22, -110.97),
    "WH-SANJOSE":   (37.34, -121.89),
    "WH-SINGAPORE": (1.35,   103.82),
    "WH-DRESDEN":   (51.05,   13.74),
}

CUOPT_ENDPOINT = os.getenv("CUOPT_ENDPOINT", "")
AVG_AIR_SPEED_KMH = 850
CUSTOMS_HANDLING_HRS = 4

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def build_travel_time_matrix(locations):
    n = len(locations)
    matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                dist = haversine_km(locations[i][0], locations[i][1], locations[j][0], locations[j][1])
                matrix[i][j] = dist / AVG_AIR_SPEED_KMH + CUSTOMS_HANDLING_HRS
    return matrix

def solve_cuopt_routing(pulp_assignments):
    if not pulp_assignments:
        return {"status": "no_assignments", "routes": []}

    all_locs = []
    loc_labels = []
    wh_set = set()
    for a in pulp_assignments:
        wh = a["warehouse"]
        if wh not in wh_set:
            wh_set.add(wh)
            coords = WH_COORDS.get(wh, (0, 0))
            all_locs.append(coords)
            loc_labels.append(f"{wh} (depot)")

    wh_list = list(wh_set)
    wh_idx = {w: i for i, w in enumerate(wh_list)}

    for a in pulp_assignments:
        dest_row = dem_df[dem_df["CASE_ID"] == a["case_id"]].iloc[0]
        all_locs.append((float(dest_row["DESTINATION_LAT"]), float(dest_row["DESTINATION_LON"])))
        loc_labels.append(f"{a['destination']} ({a['case_id']})")

    travel_matrix = build_travel_time_matrix(all_locs)

    orders = []
    for idx, a in enumerate(pulp_assignments):
        pickup_loc = wh_idx[a["warehouse"]]
        delivery_loc = len(wh_list) + idx
        sla_hrs = dem_df[dem_df["CASE_ID"] == a["case_id"]].iloc[0]["SLA_REMAINING_HRS"]
        deadline = max(24, abs(int(sla_hrs)) + 48)
        orders.append({
            "pickup_loc": pickup_loc,
            "delivery_loc": delivery_loc,
            "demand": 1,
            "earliest_pickup": 0,
            "latest_pickup": 12,
            "earliest_delivery": 0,
            "latest_delivery": deadline,
            "pickup_service_time": 2,
            "delivery_service_time": 2,
            "case_id": a["case_id"],
            "warehouse": a["warehouse"],
        })

    vehicles = [
        {"depot_loc": wh_idx[w], "capacity": 2, "earliest": 0, "latest": 200}
        for w in wh_list
    ]

    if CUOPT_ENDPOINT:
        try:
            import requests as req_lib
            payload = {
                "cost_matrix": travel_matrix.tolist(),
                "orders": [{k: v for k, v in o.items() if k not in ("case_id","warehouse")} for o in orders],
                "vehicles": vehicles,
                "time_limit_sec": 5,
            }
            resp = req_lib.post(f"{CUOPT_ENDPOINT}/solve", json=payload, timeout=30)
            if resp.status_code == 200:
                return resp.json()
        except Exception as e:
            print(f"cuOpt SPCS call failed: {e}, using CPU heuristic")

    routes = []
    for vi, wh in enumerate(wh_list):
        wh_orders = [o for o in orders if o["warehouse"] == wh]
        if not wh_orders:
            continue
        wh_orders.sort(key=lambda o: o["latest_delivery"])
        stops = []
        total_time = 0.0
        current_loc = wh_idx[wh]
        for o in wh_orders:
            p = o["pickup_loc"]
            d = o["delivery_loc"]
            total_time += travel_matrix[current_loc][p] + 2
            total_time += travel_matrix[p][d] + 2
            stops.append({
                "case_id": o["case_id"],
                "pickup": loc_labels[p],
                "delivery": loc_labels[d],
                "arrival_hr": round(total_time, 1),
                "deadline_hr": o["latest_delivery"],
            })
            current_loc = d
        routes.append({
            "vehicle_id": vi,
            "warehouse": wh,
            "total_time_hrs": round(total_time, 1),
            "stops": stops,
        })

    return {
        "status": "heuristic_cpu" if not CUOPT_ENDPOINT else "cuopt_gpu",
        "total_time": sum(r["total_time_hrs"] for r in routes),
        "routes": routes,
        "locations": loc_labels,
        "travel_matrix": travel_matrix.tolist(),
    }

print("cuOpt routing solver ready")

## Cell 6: Visualization Functions

In [ ]:
def display_allocation_table(results):
    if not results["assignments"]:
        display(HTML("<h4 style='color:red'>No feasible assignments found</h4>"))
        return
    df = pd.DataFrame(results["assignments"])
    df = df.rename(columns={
        "warehouse_city": "From", "destination": "To",
        "customer": "Customer", "severity": "Sev",
        "landed_cost": "Landed Cost ($)", "shipping_days": "Days",
        "case_id": "Case"
    })
    df = df[["Case", "Customer", "Sev", "From", "To", "Landed Cost ($)", "Days"]]
    df["Landed Cost ($)"] = df["Landed Cost ($)"].apply(lambda x: f"${x:,.0f}")
    display(HTML("<h3>Optimal Warehouse-to-Case Assignments</h3>"))
    display(df.style.set_properties(**{"text-align": "center"}).hide(axis="index"))

def display_transfers(results):
    if not results["transfers"]:
        display(HTML("<p><em>No inter-warehouse transfers needed</em></p>"))
        return
    df = pd.DataFrame(results["transfers"])
    wh_names = dict(zip(inv_df["WAREHOUSE_ID"], inv_df["WAREHOUSE_CITY"]))
    df["from"] = df["from"].map(wh_names)
    df["to"] = df["to"].map(wh_names)
    df = df.rename(columns={"from": "From", "to": "To", "quantity": "Qty", "cost": "Freight ($)"})
    df["Freight ($)"] = df["Freight ($)"].apply(lambda x: f"${x:,.0f}")
    display(HTML("<h3>Inter-Warehouse Transfers</h3>"))
    display(df.style.set_properties(**{"text-align": "center"}).hide(axis="index"))

def plot_cost_breakdown(results):
    cb = results["cost_breakdown"]
    labels = ["Shipping\n(Landed Cost)", "Inter-WH\nTransfer", "SLA Penalty\n(Unfulfilled)"]
    values = [cb["shipping"], cb["transfer"], cb["sla_penalty"]]
    colors = ["#29B6F6", "#66BB6A", "#EF5350"]

    fig = go.Figure(go.Bar(
        x=labels, y=values,
        marker_color=colors,
        text=[f"${v:,.0f}" for v in values],
        textposition="outside",
    ))
    fig.update_layout(
        title=f"Cost Breakdown — Total: ${results['total_cost']:,.0f}",
        yaxis_title="Cost ($)", template="plotly_dark",
        height=400, margin=dict(t=60, b=40),
    )
    fig.show()

def plot_sankey(results):
    wh_names = dict(zip(inv_df["WAREHOUSE_ID"], inv_df["WAREHOUSE_CITY"]))
    labels = [wh_names.get(w, w) for w in warehouses]
    dest_cities = []
    for a in results["assignments"]:
        dest = f"{a['destination']} ({a['customer']})"
        if dest not in dest_cities:
            dest_cities.append(dest)
    labels += dest_cities

    wh_idx_map = {w: i for i, w in enumerate(warehouses)}
    dest_idx_map = {d: i + len(warehouses) for i, d in enumerate(dest_cities)}

    sources, targets, values, link_colors = [], [], [], []

    for t in results["transfers"]:
        sources.append(wh_idx_map[t["from"]])
        targets.append(wh_idx_map[t["to"]])
        values.append(t["quantity"])
        link_colors.append("rgba(255,193,7,0.5)")

    for a in results["assignments"]:
        dest_label = f"{a['destination']} ({a['customer']})"
        sources.append(wh_idx_map[a["warehouse"]])
        targets.append(dest_idx_map[dest_label])
        values.append(1)
        sev_color = {"SEV1": "rgba(239,83,80,0.6)", "SEV2": "rgba(255,167,38,0.6)", "SEV3": "rgba(41,182,246,0.6)"}
        link_colors.append(sev_color.get(a["severity"], "rgba(150,150,150,0.5)"))

    node_colors = ["#29B6F6"] * len(warehouses) + ["#66BB6A"] * len(dest_cities)

    fig = go.Figure(go.Sankey(
        node=dict(pad=20, thickness=20, label=labels, color=node_colors),
        link=dict(source=sources, target=targets, value=values, color=link_colors),
    ))
    fig.update_layout(
        title="Inventory Flow: Warehouses → Fab Sites",
        template="plotly_dark", height=450, margin=dict(t=50, b=30),
    )
    fig.show()

def plot_dispatch_schedule(routing_results):
    if not routing_results.get("routes"):
        display(HTML("<p><em>No routing results</em></p>"))
        return
    fig = go.Figure()
    colors = ["#29B6F6", "#66BB6A", "#EF5350", "#FFA726"]
    for ri, route in enumerate(routing_results["routes"]):
        wh_names_map = dict(zip(inv_df["WAREHOUSE_ID"], inv_df["WAREHOUSE_CITY"]))
        wh_label = wh_names_map.get(route["warehouse"], route["warehouse"])
        for stop in route["stops"]:
            fig.add_trace(go.Bar(
                x=[stop["arrival_hr"]],
                y=[f"{wh_label} Carrier"],
                orientation="h",
                name=stop["case_id"],
                marker_color=colors[ri % len(colors)],
                text=f"{stop['case_id']} → {stop['delivery'].split('(')[0].strip()}",
                textposition="inside",
                hovertemplate=f"Case: {stop['case_id']}<br>Delivery: {stop['delivery']}<br>Arrival: {stop['arrival_hr']}h<br>Deadline: {stop['deadline_hr']}h",
                showlegend=False,
                width=0.4,
            ))
            fig.add_trace(go.Scatter(
                x=[stop["deadline_hr"]], y=[f"{wh_label} Carrier"],
                mode="markers", marker=dict(symbol="diamond", size=12, color="red"),
                name=f"SLA Deadline", showlegend=False,
                hovertemplate=f"SLA Deadline: {stop['deadline_hr']}h",
            ))
    fig.update_layout(
        title="Dispatch Schedule — Delivery Timeline (hours)",
        xaxis_title="Hours from Now", template="plotly_dark",
        height=300, margin=dict(t=50, b=40), barmode="stack",
    )
    fig.show()

def display_summary(results, routing_results):
    fulfilled = len(results["assignments"])
    total = len(cases)
    missed = len(results["unfulfilled"])
    total_cost = results["total_cost"]
    html = f"""
    <div style="display:flex; gap:20px; margin:10px 0;">
        <div style="background:#1a237e; padding:16px 24px; border-radius:8px; text-align:center;">
            <div style="font-size:28px; font-weight:bold; color:#29B6F6;">${total_cost:,.0f}</div>
            <div style="color:#90CAF9; font-size:12px;">Total Optimized Cost</div>
        </div>
        <div style="background:#1b5e20; padding:16px 24px; border-radius:8px; text-align:center;">
            <div style="font-size:28px; font-weight:bold; color:#66BB6A;">{fulfilled}/{total}</div>
            <div style="color:#A5D6A7; font-size:12px;">Cases Fulfilled</div>
        </div>
        <div style="background:#b71c1c; padding:16px 24px; border-radius:8px; text-align:center;">
            <div style="font-size:28px; font-weight:bold; color:#EF5350;">{missed}</div>
            <div style="color:#EF9A9A; font-size:12px;">Unfulfilled (Insufficient Stock)</div>
        </div>
        <div style="background:#4a148c; padding:16px 24px; border-radius:8px; text-align:center;">
            <div style="font-size:28px; font-weight:bold; color:#CE93D8;">{results['sla_coverage']}%</div>
            <div style="color:#E1BEE7; font-size:12px;">SLA Coverage</div>
        </div>
    </div>
    """
    display(HTML(html))

print("Visualization functions ready")

## Cell 7: Impact Analysis — Sensitivity & Pareto

In [ ]:
def run_pareto_sweep(base_params, steps=8):
    sweep_results = []
    for penalty_mult in np.linspace(0.2, 3.0, steps):
        p = base_params.copy()
        p["sla_sev1_penalty"] = base_params["sla_sev1_penalty"] * penalty_mult
        p["sla_sev2_penalty"] = base_params["sla_sev2_penalty"] * penalty_mult
        p["sla_sev3_penalty"] = base_params["sla_sev3_penalty"] * penalty_mult
        r = solve_pulp(p)
        sweep_results.append({
            "penalty_mult": round(penalty_mult, 2),
            "total_cost": r["total_cost"],
            "sla_coverage": r["sla_coverage"],
            "shipping_cost": r["cost_breakdown"]["shipping"],
            "fulfilled": len(r["assignments"]),
        })
    return sweep_results

def plot_pareto(sweep_results):
    df = pd.DataFrame(sweep_results)
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(
        x=df["sla_coverage"], y=df["total_cost"],
        mode="lines+markers+text",
        marker=dict(size=12, color=df["penalty_mult"], colorscale="Viridis", showscale=True,
                    colorbar=dict(title="Penalty<br>Mult")),
        text=[f"{r['penalty_mult']}x" for _, r in df.iterrows()],
        textposition="top center",
        name="Scenarios",
    ), secondary_y=False)
    fig.update_layout(
        title="Pareto Frontier: Total Cost vs SLA Coverage",
        xaxis_title="SLA Coverage (%)", template="plotly_dark",
        height=400, margin=dict(t=60, b=40),
    )
    fig.update_yaxes(title_text="Total Cost ($)", secondary_y=False)
    fig.show()

def run_sensitivity_tornado(base_params):
    base_result = solve_pulp(base_params)
    base_cost = base_result["total_cost"]

    factors = {
        "SEV1 Penalty": ("sla_sev1_penalty", 0.5, 2.0),
        "SEV2 Penalty": ("sla_sev2_penalty", 0.5, 2.0),
        "SEV3 Penalty": ("sla_sev3_penalty", 0.5, 2.0),
        "Shipping Cost": ("shipping_multiplier", 0.5, 2.0),
        "Transfer Cost": ("transfer_cost_multiplier", 0.5, 2.0),
        "Safety Stock": ("safety_stock", 0, 2),
    }

    tornado_data = []
    for label, (key, low_val, high_val) in factors.items():
        p_low = base_params.copy()
        p_low[key] = low_val if key != "safety_stock" else low_val
        if key in ("sla_sev1_penalty", "sla_sev2_penalty", "sla_sev3_penalty"):
            p_low[key] = base_params[key] * low_val
        r_low = solve_pulp(p_low)

        p_high = base_params.copy()
        p_high[key] = high_val if key != "safety_stock" else high_val
        if key in ("sla_sev1_penalty", "sla_sev2_penalty", "sla_sev3_penalty"):
            p_high[key] = base_params[key] * high_val
        r_high = solve_pulp(p_high)

        tornado_data.append({
            "factor": label,
            "low_cost": r_low["total_cost"],
            "high_cost": r_high["total_cost"],
            "delta_low": r_low["total_cost"] - base_cost,
            "delta_high": r_high["total_cost"] - base_cost,
        })

    tornado_data.sort(key=lambda x: abs(x["delta_high"] - x["delta_low"]), reverse=True)
    return tornado_data, base_cost

def plot_tornado(tornado_data, base_cost):
    fig = go.Figure()
    labels = [d["factor"] for d in tornado_data]
    low_deltas = [d["delta_low"] for d in tornado_data]
    high_deltas = [d["delta_high"] for d in tornado_data]

    fig.add_trace(go.Bar(
        y=labels, x=low_deltas, orientation="h",
        name="Low Scenario", marker_color="#29B6F6",
        text=[f"${d:+,.0f}" for d in low_deltas], textposition="outside",
    ))
    fig.add_trace(go.Bar(
        y=labels, x=high_deltas, orientation="h",
        name="High Scenario", marker_color="#EF5350",
        text=[f"${d:+,.0f}" for d in high_deltas], textposition="outside",
    ))
    fig.add_vline(x=0, line_dash="dash", line_color="white")
    fig.update_layout(
        title=f"Sensitivity Tornado — Base Cost: ${base_cost:,.0f}",
        xaxis_title="Cost Delta ($)", template="plotly_dark",
        height=400, barmode="overlay", margin=dict(t=60, b=40, l=120),
    )
    fig.show()

print("Impact analysis functions ready")

## Cell 8: Orchestrator — Run Optimization Pipeline

In [ ]:
last_results = {}
last_routing = {}
last_params = {}

def get_current_params():
    return {
        "sla_sev1_penalty": sla_sev1_penalty.value,
        "sla_sev2_penalty": sla_sev2_penalty.value,
        "sla_sev3_penalty": sla_sev3_penalty.value,
        "shipping_multiplier": shipping_multiplier.value,
        "transfer_cost_multiplier": transfer_cost_multiplier.value,
        "customer_priority": customer_priority.value,
        "safety_stock": safety_stock.value,
        "batch_exclusion": batch_exclusion.value,
    }

def on_run_click(btn):
    global last_results, last_routing, last_params
    with output_area:
        clear_output(wait=True)
        params = get_current_params()
        last_params = params

        display(HTML("<h2>Optimizing...</h2>"))
        display(HTML("<p>Stage 1: PuLP MIP — solving warehouse allocation...</p>"))

        results = solve_pulp(params)
        last_results = results

        if results["status"] != "Optimal":
            display(HTML(f"<h3 style='color:red'>Solver Status: {results['status']}</h3>"))
            display(HTML("<p>Try relaxing constraints (reduce safety stock or disable batch exclusion)</p>"))
            return

        display(HTML("<p>Stage 2: cuOpt VRP — solving delivery routes...</p>"))
        routing = solve_cuopt_routing(results["assignments"])
        last_routing = routing

        clear_output(wait=True)

        display(HTML("<h2>Optimization Results</h2>"))
        display_summary(results, routing)
        display_allocation_table(results)
        display_transfers(results)
        plot_cost_breakdown(results)
        plot_sankey(results)
        plot_dispatch_schedule(routing)

        display(HTML("<h2>Impact Analysis</h2>"))
        display(HTML("<p><em>Running sensitivity sweep...</em></p>"))

        sweep = run_pareto_sweep(params)
        plot_pareto(sweep)

        tornado_data, base_cost = run_sensitivity_tornado(params)
        plot_tornado(tornado_data, base_cost)

        display(HTML(f"""
        <div style='margin-top:16px; padding:12px; background:#263238; border-radius:8px; border-left:4px solid #29B6F6;'>
            <b>Solver:</b> PuLP CBC + {'cuOpt GPU' if CUOPT_ENDPOINT else 'CPU Heuristic'}<br>
            <b>Status:</b> {results['status']}<br>
            <b>Timestamp:</b> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}<br>
            <b>Scenario Hash:</b> {uuid.uuid4().hex[:8]}
        </div>
        """))

run_button.on_click(on_run_click)
display(output_area)
print("Click 'Run Optimization' above to execute the pipeline")

## Cell 9: Write-Back Results to Snowflake

In [ ]:
def save_scenario_to_snowflake(params, results, routing):
    if not SF_AVAILABLE:
        print("Snowflake connector not available — skipping write-back")
        return None
    try:
        conn = snowflake.connector.connect(
            connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "mydemo"
        )
        cur = conn.cursor()

        scenario_id = uuid.uuid4().hex[:12]
        params_json = json.dumps(params)
        cur.execute(f"""
            INSERT INTO KLA_SUPPLY_CHAIN.APP.OPT_SCENARIOS (SCENARIO_ID, LABEL, PARAMS_JSON)
            SELECT '{scenario_id}', 'Interactive Run', PARSE_JSON('{params_json}')
        """)

        run_id = uuid.uuid4().hex[:12]
        assignments_json = json.dumps(results.get("assignments", []))
        transfers_json = json.dumps(results.get("transfers", []))
        routes_json = json.dumps(routing.get("routes", []))
        cur.execute(f"""
            INSERT INTO KLA_SUPPLY_CHAIN.APP.OPT_RESULTS
            (RUN_ID, SCENARIO_ID, TOTAL_COST, SLA_COVERAGE, ASSIGNMENTS_JSON, TRANSFERS_JSON, ROUTES_JSON)
            SELECT '{run_id}', '{scenario_id}', {results['total_cost']}, {results['sla_coverage']},
            PARSE_JSON('{assignments_json}'), PARSE_JSON('{transfers_json}'), PARSE_JSON('{routes_json}')
        """)

        conn.commit()
        cur.close()
        conn.close()
        print(f"Saved to Snowflake — Scenario: {scenario_id}, Run: {run_id}")
        return scenario_id
    except Exception as e:
        print(f"Write-back failed: {e}")
        return None

save_button = widgets.Button(
    description="  Save to Snowflake", button_style="info", icon="cloud-upload",
    layout=widgets.Layout(width="220px", height="40px"),
)

save_output = widgets.Output()

def on_save_click(btn):
    with save_output:
        clear_output(wait=True)
        if not last_results:
            print("No results to save — run optimization first")
            return
        save_scenario_to_snowflake(last_params, last_results, last_routing)

save_button.on_click(on_save_click)
display(widgets.HBox([save_button]))
display(save_output)